In [1]:
import numpy as np
import pandas as pd

In [2]:
test_user_movie_df = pd.read_parquet("../data/processed/test_user_movie_df.parquet")
existing_user_top_n_recommendations = pd.read_parquet("../data/processed/existing_user_top_n_recommendations.parquet")

In [3]:
existing_user_top_n_recommendations.head(3)

,userid,movieid,title,weighted_rating
0,1,318,"Shawshank Redemption, The (1994)",4.384051
1,1,858,"Godfather, The (1972)",4.281711
2,1,50,"Usual Suspects, The (1995)",4.246032


In [4]:
test_user_movie_df.loc[(test_user_movie_df["userid"] == 1) & (test_user_movie_df["rating"] >= 4)].head()

,userid,movieid,rating,title,genres
2,1,8360,4.0,Shrek 2 (2004),Adventure|Animation|Children|Comedy|Musical|Ro...
3,1,4973,4.5,"Amelie (Fabuleux destin d'Amélie Poulain, Le) ...",Comedy|Romance
7,1,8154,5.0,"Dolce Vita, La (1960)",Drama
8,1,6016,5.0,City of God (Cidade de Deus) (2002),Action|Adventure|Crime|Drama|Thriller
10,1,3448,4.0,"Good Morning, Vietnam (1987)",Comedy|Drama|War


In [5]:
test_user_movie_df.loc[(test_user_movie_df["userid"] == 3) & (test_user_movie_df["rating"] >= 4)].head()

,userid,movieid,rating,title,genres
51,3,106920,4.0,Her (2013),Drama|Romance|Sci-Fi
52,3,4963,4.0,Ocean's Eleven (2001),Crime|Thriller
53,3,5959,4.0,Narc (2002),Crime|Drama|Thriller
55,3,166528,4.0,Rogue One: A Star Wars Story (2016),Action|Adventure|Fantasy|Sci-Fi
58,3,33794,4.0,Batman Begins (2005),Action|Crime|IMAX


In [6]:
#DataPrep for Metrics Calculation
test_user_liked_movie_df = (
    test_user_movie_df.loc[
        test_user_movie_df["rating"] >= 4.0,
        ["userid", "movieid", "title"]
    ]
)

liked_movies_by_user = (
    test_user_liked_movie_df
    .groupby("userid")["movieid"]
    .apply(set)
    .to_dict()
)

recommendations_to_user = (
    existing_user_top_n_recommendations
    .groupby("userid")["movieid"]
    .apply(list)
    .to_dict()
)

In [7]:
#PRECISION Calculation
precision_scores = []
k = 10

for userid in liked_movies_by_user:

    hits = len(
        liked_movies_by_user[userid] &
        set(recommendations_to_user[userid])
    )

    precision_scores.append(hits / k)

In [8]:
model_precision = np.mean(precision_scores)

print(f"Mean Precision@10    : {np.mean(precision_scores):.2%}")
print(f"Median Precision@10  : {np.median(precision_scores):.2%}")
print(f"25th Percentile      : {np.percentile(precision_scores, 25):.2%}")
print(f"75th Percentile      : {np.percentile(precision_scores, 75):.2%}")
print(f"Minimum Precision@10 : {np.min(precision_scores):.2%}")
print(f"Maximum Precision@10 : {np.max(precision_scores):.2%}")

Mean Precision@10    : 8.24%
Median Precision@10  : 0.00%
25th Percentile      : 0.00%
75th Percentile      : 10.00%
Minimum Precision@10 : 0.00%
Maximum Precision@10 : 100.00%


In [9]:
#RECALL Calculation
recall_scores = []

for userid in liked_movies_by_user:

    hits = len(
        liked_movies_by_user[userid] &
        set(recommendations_to_user[userid])
    )

    if liked_movies_by_user[userid]:
      recall_scores.append(hits / len(liked_movies_by_user[userid]))

In [10]:
model_recall = np.mean(recall_scores)
print(f"Model Recall: {model_recall:.2%}")
print("REcall is lesser than precision as avg user has ~16 liked movies in the test set, hence it is calculated at base 16")

Model Recall: 6.75%
REcall is lesser than precision as avg user has ~16 liked movies in the test set, hence it is calculated at base 16


In [11]:
print(f"Mean Recall@10    : {np.mean(recall_scores):.2%}")
print(f"Median Recall@10  : {np.median(recall_scores):.2%}")
print(f"25th Percentile      : {np.percentile(recall_scores, 25):.2%}")
print(f"75th Percentile      : {np.percentile(recall_scores, 75):.2%}")
print(f"Minimum Recall@10 : {np.min(recall_scores):.2%}")
print(f"Maximum Recall@10 : {np.max(recall_scores):.2%}")

Mean Recall@10    : 6.75%
Median Recall@10  : 0.00%
25th Percentile      : 0.00%
75th Percentile      : 9.09%
Minimum Recall@10 : 0.00%
Maximum Recall@10 : 100.00%


In [ ]:
#NDCG Calculation

k=10
ndcg_scores = []

for userid in liked_movies_by_user.keys():

    relevant_movies = liked_movies_by_user[userid]
    recommended_movies = recommendations_to_user.get(userid, [])[:k]

    dcg = 0

    for rank, movieid in enumerate(recommended_movies, start=1):

        if movieid in relevant_movies:
            dcg += 1 / np.log2(rank + 1)

    ideal_hits = min(len(relevant_movies), k)

    idcg = sum(
        1 / np.log2(rank + 1)
        for rank in range(1, ideal_hits + 1)
    )

    if idcg > 0:
        ndcg_scores.append(dcg / idcg)

model_ndcg = np.mean(ndcg_scores)
print("Model NDCG@10 :", model_ndcg * 100)
print("Higher NDCG than precision as it rewards earlier hits i.e. rank order of the recommended movies")

Model %NDCG@10 : 11.032188228264543
Higher NDCG than precision as it rewards earlier hits i.e. rank order of the recommended movies


In [13]:
evaluation_summary = pd.DataFrame({
    "Metric": [
        "NDCG@10",
        "Precision@10",
        "Recall@10",
        "Mean Relevant Movies/User",
        "Users Evaluated"
    ],
    "Value": [
        f"{model_ndcg:.2%}",
        f"{model_precision:.2%}",
        f"{model_recall:.2%}",
        round(test_user_liked_movie_df.groupby("userid")["movieid"].count().mean(), 2),
        f"{test_user_liked_movie_df['userid'].nunique():,}"
    ]
})

display(evaluation_summary)

,Metric,Value
0,NDCG@10,11.03%
1,Precision@10,8.24%
2,Recall@10,6.75%
3,Mean Relevant Movies/User,15.77
4,Users Evaluated,"160,259"
